# CP2 - PySpark: Primeros Pasos

### Antes de Empezar...

Si has cerrado el proyecto o has abierto `Nueva Ventana` en VS Code, pulsa en el primer icono del menú de la izquierda `Explorador` y después en `Abrir Carpeta`. 

Busca la ruta en la que guardas el repositorio de GitHub en tu PC, normalmente *`Documentos => GitHub => fundamentos_big_data` * y pulsa en `Abrir`. De esta forma, ya tendrás la carpeta de tu repositorio cargada en VS Code.

> ¡Importante! Antes de abrir esa carpeta, debes ir a GitHub Desktop y realizar un `pull` para asegurarte de que tus archivos están actualizados. También puedes hacer el pull desde VS Code.  

### Requisitos para arrancar el entonro
- Iniciar Docker (abrir docker desktop).
- Iniciar GitHub (abrir github desktop) y estar logueado.
- En el terminal, ejecutar el siguiente comando en la raíz del proyecto `fundamentos_big_data %` 
```bash
docker-compose up --build
```
> ¡NO TE OLVIDES! Docker debe estar abierto y en ejecución en tu pc para poder utilizarlo. 


### Seleccionar Kernel
--> Seleccionar kernel\
--> Servidor de Juypter existente\
--> Escriba la dirección URL del servidor de Jupyter en ejecución\
--> Seleccionar http://127.0.0.1:8888/?token=xxxxxxxx y pulsar Enter\
--> 127.0.0.1 y pulsar Enter\
--> Seleccionar Python 3 (ipykernel)

___________________

## Objetivos de la sesión

- Comprender qué es PySpark y por qué se usa en entornos Big Data.
- Crear una SparkSession en local.
- Realizar la lectura y exploración básica de datasets en formatos CSV y JSON.
- Entender conceptos clave: ejecución distribuida, Lazy Evaluation, coalesce/repartition, etc.

Fuente: [Medium - Conceptos Clave de Pyspark](https://medium.com/@alonso.md/spark-conceptos-claves-33d0db88db8)


## 1. ¿Qué es PySpark?

PySpark es la **API de Python para Apache Spark**, un motor de procesamiento distribuido de datos a gran escala. Spark permite trabajar con **datasets enormes** que no caben en memoria en un solo ordenador, **repartiendo los cálculos entre múltiples nodos**. PySpark combina:
- La escalabilidad de Spark.
- La simplicidad de Python.

### 1.1 Conceptos clave de PySpark

1. **Componentes de Spark**

   * **Spark Core**: motor base, gestiona memoria, fallos y tareas.
   * **Spark SQL**: consultas con SQL y soporte para formatos (Parquet, JSON, Hive).
   * **Spark Streaming**: datos en tiempo real (Kafka, logs, etc.).
   * **MLlib**: librería de machine learning.
   * **GraphX**: análisis de grafos (redes sociales, conexiones).

<p align="center">
  <img src="https://miro.medium.com/v2/format:webp/1*UfSqtksZQ_vgrrQtK5DPRA.png" alt="componentes pyspark" width="800" height="500">
</p>

- Map → trocea los datos y saca pares clave-valor.
- Reduce → agrupa por clave y resume el resultado.

> Imagina que 100 alumnos cuentan palabras en diferentes libros (Map) y luego el profe junta todos los resultados para sacar el total (Reduce).

2. **Estructuras de datos**

   * **RDD**: colección distribuida, bajo nivel.
   * **DataFrame**: tabla distribuida con esquema, más optimizada.
   * **Dataset**: mezcla de DataFrame y RDD (solo Scala/Java).
     👉 En PySpark usamos principalmente **DataFrames**.

3. **Planificación de consultas (Catalyst Optimizer)**

   * Spark transforma nuestro código en un **plan lógico** → lo optimiza → genera un **plan físico** → lo convierte a código ejecutable.
   * Esto permite que Spark elija el plan más eficiente (ej. broadcast join vs shuffle join).

4. **Gestión de memoria**

   * Spark divide la memoria entre:

     * **Driver** (coordinador).
     * **Executors** (trabajadores).
   * Hay memoria para ejecución (cálculos) y para almacenamiento (cache de datos).

<p align="center">
  <img src="https://miro.medium.com/v2/resize:fit:720/format:webp/1*k4iWNgiuZdaXQiuiAukaaQ.png" alt="memoria pyspark" width="800" height="600">
</p>

5. **Transformaciones y acciones**

   * **Transformaciones**: `filter`, `select`, `groupBy`. No ejecutan nada aún (lazy evaluation).
   * **Acciones**: `count`, `collect`, `show`. Disparan las transformaciones y por tanto, realizan la ejecución real.

6. **Optimización**

   * **Adaptive Query Execution (AQE)**: Spark ajusta en tiempo real el plan según los datos.
   * **Partitioning, Coalesce y Repartition**: controlan cómo se dividen los datos.
   * **Caching/Persist**: evita recomputar datos repetidamente.

<p align="center">
  <img src="http://mamel.es/wp-content/uploads/2017/08/MapReduceWordCountOverview1.png" alt="map reduce" width="800" height="300">
</p>

- Map → trocea los datos y saca pares clave-valor.
- Reduce → agrupa por clave y resume el resultado.

> Imagina que 100 alumnos cuentan palabras en diferentes libros (Map) y luego el profe junta todos los resultados para sacar el total (Reduce).


7. **Variables compartidas**

   * **Broadcast Variables**: envían copias pequeñas de datos a todos los nodos.
   * **Accumulators**: permiten sumar contadores globales en paralelo.

---

👉 **CONCEPTO CLAVE =>** Spark es como un *“orquestador de datos”*:

* Divide el dataset.
* Reparte el trabajo.
* Optimiza la ejecución.
* Controla la memoria.

> PySpark no es solo “otro pandas”, sino un **motor distribuido inteligente**.

## 2. Ejecuciones de Prueba

Antes de empezar a trabajar con pyspark en profundidad, vamos a realizar unas ejecuciones de prueba para confirmar que todo funciona correctamente y que podemos lanzar pyspark con normalidad. 

### 2.1 Crear una `SparkSession`

In [1]:
from pyspark.sql import SparkSession

# Crear la sesión de Spark
spark = SparkSession.builder \
    .appName("Pract1_PySpark") \
    .getOrCreate()

# Verificar la versión
print("Versión de Spark:", spark.version)

Versión de Spark: 3.5.0


In [2]:
# Datos de ejemplo
data = [("Alice", 34), ("Bob", 45), ("Carmen", 29)]
df = spark.createDataFrame(data, ["Nombre", "Edad"])
df.show()

+------+----+
|Nombre|Edad|
+------+----+
| Alice|  34|
|   Bob|  45|
|Carmen|  29|
+------+----+



### 2.2 Importar Librerías Fundamentales de PySpark

In [3]:
# Funciones, tipos y ventanas de PySpark
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window

### 2.3 Ejemplo de DataFrames

Vamos a crear un par de DataFrames: `ventas` y `clientes`. 

In [4]:
## Construimos los DataFrames de ejemplo
ventas = [
    {"pedido_id": 1,  "cliente_id": 101, "ciudad": "Valencia", "categoria": "Electrónica", "importe": 199.99},
    {"pedido_id": 2,  "cliente_id": 102, "ciudad": "Madrid",   "categoria": "Hogar",        "importe": 89.50},
    {"pedido_id": 3,  "cliente_id": 101, "ciudad": "Valencia", "categoria": "Hogar",        "importe": 45.00},
    {"pedido_id": 4,  "cliente_id": 103, "ciudad": "Sevilla",  "categoria": "Electrónica", "importe": 399.00},
    {"pedido_id": 5,  "cliente_id": 104, "ciudad": "Madrid",   "categoria": "Moda",         "importe": 59.99},
    {"pedido_id": 6,  "cliente_id": 102, "ciudad": "Madrid",   "categoria": "Electrónica", "importe": 129.90},
    {"pedido_id": 7,  "cliente_id": 105, "ciudad": "Valencia", "categoria": "Moda",         "importe": 75.00},
    {"pedido_id": 8,  "cliente_id": 103, "ciudad": "Sevilla",  "categoria": "Hogar",        "importe": 34.99},
    {"pedido_id": 9,  "cliente_id": 106, "ciudad": "Bilbao",   "categoria": "Electrónica", "importe": 249.00},
    {"pedido_id": 10, "cliente_id": 106, "ciudad": "Bilbao",   "categoria": "Moda",         "importe": 39.90},
]

clientes = [
    {"cliente_id": 101, "nombre": "Ana",   "edad": 34, "alta": "2022-03-01"},
    {"cliente_id": 102, "nombre": "Luis",  "edad": 41, "alta": "2021-11-12"},
    {"cliente_id": 103, "nombre": "Sara",  "edad": 29, "alta": "2023-06-23"},
    {"cliente_id": 104, "nombre": "Mario", "edad": 37, "alta": "2022-08-05"},
    {"cliente_id": 105, "nombre": "Irene", "edad": 25, "alta": "2024-01-17"},
    {"cliente_id": 106, "nombre": "Pablo", "edad": 45, "alta": "2020-09-30"},
]

In [5]:
## Los convertimos en DataFrames de PySpark
ventas_df = spark.createDataFrame(ventas)
clientes_df = spark.createDataFrame(clientes)

In [6]:
## Mostramos el esquema de los DataFrames
ventas_df.printSchema(); clientes_df.printSchema()

root
 |-- categoria: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- cliente_id: long (nullable = true)
 |-- importe: double (nullable = true)
 |-- pedido_id: long (nullable = true)

root
 |-- alta: string (nullable = true)
 |-- cliente_id: long (nullable = true)
 |-- edad: long (nullable = true)
 |-- nombre: string (nullable = true)



In [7]:
## Mostramos las primeras filas del DataFrame de ventas
ventas_df.show(5,0)

+-----------+--------+----------+-------+---------+
|categoria  |ciudad  |cliente_id|importe|pedido_id|
+-----------+--------+----------+-------+---------+
|Electrónica|Valencia|101       |199.99 |1        |
|Hogar      |Madrid  |102       |89.5   |2        |
|Hogar      |Valencia|101       |45.0   |3        |
|Electrónica|Sevilla |103       |399.0  |4        |
|Moda       |Madrid  |104       |59.99  |5        |
+-----------+--------+----------+-------+---------+
only showing top 5 rows



In [8]:
## Mostramos las primeras filas del DataFrame de clientes
clientes_df.show()

+----------+----------+----+------+
|      alta|cliente_id|edad|nombre|
+----------+----------+----+------+
|2022-03-01|       101|  34|   Ana|
|2021-11-12|       102|  41|  Luis|
|2023-06-23|       103|  29|  Sara|
|2022-08-05|       104|  37| Mario|
|2024-01-17|       105|  25| Irene|
|2020-09-30|       106|  45| Pablo|
+----------+----------+----+------+



In [9]:
## Agrupamos las ventas por ciudad y calculamos algunas métricas
agg_df = (ventas_df
      .groupBy("ciudad")
      .agg(
          F.round(F.sum("importe"), 2).alias("importe_total"),
          F.round(F.avg("importe"), 2).alias("ticket_medio"),
          F.countDistinct("cliente_id").alias("num_clientes")
      )
)
agg_df.orderBy("importe_total", ascending=False).show(10,0)
#agg_df.explain(True)

+--------+-------------+------------+------------+
|ciudad  |importe_total|ticket_medio|num_clientes|
+--------+-------------+------------+------------+
|Sevilla |433.99       |217.0       |1           |
|Valencia|319.99       |106.66      |2           |
|Bilbao  |288.9        |144.45      |1           |
|Madrid  |279.39       |93.13       |2           |
+--------+-------------+------------+------------+



<p align="center">
  <img src="http://miro.medium.com/v2/0*-Y8lKRffc5K6Cvxx.png" alt="memoria pyspark" width="500" height="350">
</p>


In [10]:
## Unimos los DataFrames de ventas y clientes
joined_df = ventas_df.join(
            F.broadcast(clientes_df), 
            on="cliente_id", 
            how="left")

selected_columns = ["pedido_id", "nombre", "ciudad", "categoria", "importe", "edad"]

joined_df.select(selected_columns).show(5,0)
#joined_df.explain(True)

+---------+------+--------+-----------+-------+----+
|pedido_id|nombre|ciudad  |categoria  |importe|edad|
+---------+------+--------+-----------+-------+----+
|1        |Ana   |Valencia|Electrónica|199.99 |34  |
|2        |Luis  |Madrid  |Hogar      |89.5   |41  |
|3        |Ana   |Valencia|Hogar      |45.0   |34  |
|4        |Sara  |Sevilla |Electrónica|399.0  |29  |
|5        |Mario |Madrid  |Moda       |59.99  |37  |
+---------+------+--------+-----------+-------+----+
only showing top 5 rows



## 3. Cargar Datos en PySpark

Ahora vamos a cargar datos para trabajar con ellos desde PySpark. Para ello, debes seguir los siguientes pasos:

1. Guardar este archivo (`Ctrl+S`).
2. Hacer un Commit en GitHub. 
      1. Desde **GitHub Desktop**: en la ventana principal, en la parte de abajo, escribe un nombre para la actualización y una pequeña descripción (Ej,: Actualización practica 2 v1.0. Descripción: cambios guardados del archivo). 
      2. Desde **VS Code (NO RECOMENDADO, puede fallar)**: en el menú izquierdo, el 3º icono (control de código fuente). Te aparecerá el notebook y un icono con un `+`. Pulsa en el mas para añdir a `staged` y escribe un mensaje. Luego confirma y persiste los cambios. 
2. Descarga el archivo `data_spark.zip` desde canvas.
3. Abre la carpeta de tu pc de GitHub (normalmente: `documents\fundamentos_big_data`). Allí, entra en la carpeta llamada `notebooks` y mete el archivo `data_spark.zip` en ella. 
4. Accede a GitHub Desktop para realizar un commit y un `push` para actualizar el repositorio. 
____________

Abre un nuevo terminal (pulsando en `+` en el menú del terminal):

Para MAC, Linux o WSL:

```bash
RUTA=$(pwd)/notebooks/data_practicas_1 && mkdir -p "$RUTA" && tar -xzvf notebooks/data_spark.zip -C "$RUTA" && echo "Archivos extraídos en: $RUTA"
```
Para Windows (PowerShell):

```powershell
$RUTA = "$($PWD.Path)\notebooks\data_practicas_1"; `
New-Item -ItemType Directory -Force -Path $RUTA; `
tar -xzvf notebooks/data_spark.zip -C $RUTA; `
Write-Output "Archivos extraídos en: $RUTA"
```

De esta forma, accedemos a la carpeta `notebooks` que es la que está conectada a Docker. Allí creamos una carpeta para descomprimir la data, la descomprimimos y la guardamos en la carpeta definida. El resultado de esta ejecución será la ruta donde está guardada la data. 

Una vez hecho esto, ya tendremos los datos en el contenedor de Docker, en la ruta: `/home/jovyan/work/data_practicas_1`. De esta forma, guardaremos dicha ruta en una variable llamada `DATA_PATH`.

In [14]:
DATA_PATH = "/home/jovyan/work/data_practicas_1/"

In [15]:
## Leer un CSV
spark.read.csv(
    DATA_PATH + "dataCSV.csv", 
    header=True, 
    inferSchema=True)\
.show(5,0)

+-----------+-------------+--------------------------------------------------------------+---------------------+-----------+------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+------+--------+-------------+----------------------------------------------+-----------------+----------------+----------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [16]:
## Leer un PARQUET
spark.read.parquet(DATA_PATH + "estudiantes.parquet").show(5,0)

+------+----+----+----------+
|nombre|sexo|peso|graduacion|
+------+----+----+----------+
|Jose  |M   |80  |2000      |
|Hilda |F   |50  |2000      |
|Juan  |M   |75  |2000      |
|Pedro |M   |76  |2001      |
|Katia+|F   |65  |2001      |
+------+----+----+----------+



In [17]:
df_prueba = spark.read.parquet(DATA_PATH + "estudiantes.parquet")
df_prueba.show()

+------+----+----+----------+
|nombre|sexo|peso|graduacion|
+------+----+----+----------+
|  Jose|   M|  80|      2000|
| Hilda|   F|  50|      2000|
|  Juan|   M|  75|      2000|
| Pedro|   M|  76|      2001|
|Katia+|   F|  65|      2001|
+------+----+----+----------+



## 4. RDDs y DataFrames

### 4.1 ¿Qué es un RDD?

Resilient Distributed Dataset (RDD). Colección de elementos particionados en los distintos clúster y que pueden ser operados en paralelo.

Características:

- **Dependencias**: lista que le indica a spark cómo se construye un RDD. Le permite replicar.
- **Particiones**: capacidad de dividir el trabajo entre ejecutores.
- **Función de cálculo**.

### 4.2 Crear un RDD

Lo primero que tenemos que hacer es crear un `Spark Context`, que está asociado a una `Spark Session`.

In [18]:
sc = spark.sparkContext

**Crear un RDD vacío**

In [19]:
rdd_vacio = sc.emptyRDD 

**Crear un RDD con `paralelize()`**.

Con esta función, podemos definir los elementos y el número de particiones que queremos crear. Si lo creamos vacío, simplmente definimos la lista vacía `[]`.

In [23]:
rdd_vacio2 = sc.parallelize([],3)
rdd_vacio2.getNumPartitions() # Para ver número de particiones.

rdd = sc.parallelize([1,2,3,4,5], 3)
print(rdd )
print(rdd.collect())

ParallelCollectionRDD[67] at readRDDFromFile at PythonRDD.scala:289
[1, 2, 3, 4, 5]


**Crear RDD desde un archivo de Texto**

Para cargar un RDD desde un archivo de texto:

- `.textFile(ruta)`: para partir de un archivo de texto y leerlo por líneas.
- `.wholeTextFiles(ruta)`: para partir de un archivo de texto y leerlo por completo y todo junto. Devuelve nombre del archivo y texto.

In [24]:
rdd_texto = sc.textFile( DATA_PATH + 'rdd_source.txt')
rdd_texto.collect()

['Así podemos crear', 'un RDD desde un', 'archivo de texto!!!']

In [25]:
rdd_texto_comp = sc.wholeTextFiles(DATA_PATH + 'rdd_source.txt')
rdd_texto_comp.collect()

[('file:/home/jovyan/work/data_practicas_1/rdd_source.txt',
  'Así podemos crear\nun RDD desde un\narchivo de texto!!!')]

**Crear un RDD a partir de otro existente**

Podemos usar la función `map()` con `lambda` para crear un nuevo RDD.

> Nota: como los RDD son inmutables, al modificarlos se genera un nuevo RDD, al igual que ocurre con los string, por ejemplo.

In [26]:
rdd_suma = rdd.map(lambda x: x+1)
rdd_suma.collect()

[2, 3, 4, 5, 6]

**Crear un RDD a partir de DataFrame**

In [27]:
df = spark.createDataFrame([(1,2),(2,3),(3,4)], ['id', 'número'])
rdd_df = df.rdd
rdd_df.collect()

[Row(id=1, número=2), Row(id=2, número=3), Row(id=3, número=4)]

Hay dos operaciones principales que se pueden realizar en un RDD:

- Transformaciones.
- Acciones.

### 4.3 Transformaciones en un RDD

Tenemos 3 tipos de transformaciones:

- Dividir el elemento de entrada.
- Filtrar elementos.
- Realizar cálculos.

> **Lazy evaluation**: las transformaciones no se realizan de forma instantánea, sino que se ejecuta cuando el controlador solicita algunos datos.

In [29]:
rdd1 = sc.parallelize([1,2,3,4,5])
rdd2 = sc.parallelize(['jose','juan','pedro','maria'])
rdd3 = sc.parallelize([x for x in range(10)])
rdd4 = sc.parallelize([1,2,3,4,5], 10)
rdd5 = sc.parallelize([ ('casa',2), ('parque',1), ('que',5),
                        ('casa', 1), ('escuela',2), ('casa',1),
                        ('que',1)] )

##### **Función `map()`**

Aplica una función a cada elemento del RDD y lo devuelve transformado.

In [30]:
rdd_map = rdd1.map(lambda x: x-1)
rdd_map2 = rdd2.map(lambda x: x.upper())

print(rdd_map.collect())
print(rdd_map2.collect())

[0, 1, 2, 3, 4]
['JOSE', 'JUAN', 'PEDRO', 'MARIA']


##### **Función `flatmap()`**

Realiza un map pero aplanando los datos, es decir, eliminando las tuplas.

In [31]:
rdd_square = rdd1.map(lambda x: (x, x**2))
rdd_square_flat = rdd1.flatMap(lambda x: (x, x**2))
rdd_square_flat2 = rdd2.flatMap(lambda x: (x, x.upper()))

print(rdd_square.collect())
print(rdd_square_flat.collect())
print(rdd_square_flat2.collect())

[(1, 1), (2, 4), (3, 9), (4, 16), (5, 25)]
[1, 1, 2, 4, 3, 9, 4, 16, 5, 25]
['jose', 'JOSE', 'juan', 'JUAN', 'pedro', 'PEDRO', 'maria', 'MARIA']


##### **Función `filter()`**

Nos permite filtrar resultados por condiciones.

In [32]:
rdd_filter = rdd3.filter(lambda x: x % 2 == 0)
rdd_filter2 = rdd2.filter(lambda x: x.startswith('j'))
rdd_filter3 = rdd2.filter(lambda x: x.startswith('m') and x.find('i') == 3)

print(rdd_filter.collect())
print(rdd_filter2.collect())
print(rdd_filter3.collect())

[0, 2, 4, 6, 8]
['jose', 'juan']
['maria']


##### **Función `coalesce()`**

Nos permite reducir el número de particiones y colapsarlas en un número menor.

In [36]:
print("Número de particiones:", rdd4.getNumPartitions())

Número de particiones: 10


In [37]:
rdd_coal = rdd4.coalesce(5)
print("Número de particiones:", rdd_coal.getNumPartitions())

Número de particiones: 5


##### **Función `repartition()`**

Sirve para combinar o dividir las particiones.

In [38]:
rdd_repart = rdd4.repartition(8)
print("Número de particiones:", rdd_repart.getNumPartitions())

Número de particiones: 8


##### **Función `reduceByKey()`**

Se trata de la función `reduce()` pero que solo realiza el reduce usando la clave del elemento. Para usar la función `lambda` tenemos que usar dos parámetros y realizar la operación entre claves para que funcione correctamente.

In [39]:
rdd5.collect()

rdd_reduceKey = rdd5.reduceByKey(lambda x,y: x + y)
rdd_reduceKey2 = rdd5.reduceByKey(lambda x,y: x - y)

print(rdd_reduceKey.collect())
print(rdd_reduceKey2.collect())

[('parque', 1), ('casa', 4), ('escuela', 2), ('que', 6)]
[('parque', 1), ('casa', 0), ('escuela', 2), ('que', 4)]


### 4.4 Acciones en un RDD

Las acciones son las operaciones que ejecutan el plan de cómputo y devuelven un resultado ya sea al driver o escribiéndolo en un sistema de almacenamiento externo.

Tenemos 2 tipos de acciones:

- Driver: se ejecutan en el driver y devuelve datos al controlador.
- Distributed: se ejecutan en los nodos del clúster.

> **Lazy evaluation**: las acciones no se realizan de forma instantánea, sino que se ejecuta cuando el controlador solicita algunos datos.

In [41]:
rdd1a = sc.parallelize([2,4,6,8])
rdd2a = sc.parallelize(['j','o','r','g','e'])

##### **Función `reduce()`**

In [42]:
rdd_reduce = rdd1a.reduce(lambda x,y: x+y)
rdd_reduce1 = rdd1a.reduce(lambda x,y: x*y)

print(rdd_reduce)
print(rdd_reduce1)

20
384


##### **Función `count()`**

In [43]:
rdd_count = rdd1a.count()
rdd_count2 = rdd2a.count()

print(rdd_count)
print(rdd_count2)

4
5


##### **Función `collect()`**

Devuelve todos los elementos del RDD como una lista al driver.

In [ ]:
rdd_col = rdd1a.collect()
rdd_col2 = rdd2a.collect()

print(rdd1a) # Si no usamos collect, nos muestra el RDD, no los elementos.
print(rdd_col)
print(rdd_col2)

ParallelCollectionRDD[120] at readRDDFromFile at PythonRDD.scala:289
[2, 4, 6, 8]
['j', 'o', 'r', 'g', 'e']


##### **Función `take()`**

Sirve para seleccionar los primeros `n` elementos del RDD.

In [45]:
rdd_take = rdd1a.take(2)

print(rdd1a.collect())
print(rdd_take)

[2, 4, 6, 8]
[2, 4]


##### **Función `max() / min()`**

In [46]:
rdd_max = rdd1a.max()
rdd_min = rdd1a.min()

print(rdd1a.collect())
print(rdd_max)
print(rdd_min)

[2, 4, 6, 8]
8
2


##### **Función `saveAsTextFile()`**

Nos sirve para guardar como archivo de texto. Como argumento, debemos pasarle la ruta en la que se debe guardar dicho archivo. Al ejecutarlo, vemos que lo ha guardado en varias partes (`part-00000`,  `part-000001`, etc.). Esto es debido a que tenemos varias particiones. Usando la función `coalesce()` podemos guardarlo en el mismo fichero poniendo un 1 como argumento.

> Es adecuado hacer el coalesce cuando no tenemos muchos datos.

In [48]:
rdd2a.saveAsTextFile(DATA_PATH + 'rdd_2_parts')
rdd2a.coalesce(1).saveAsTextFile(DATA_PATH + 'rdd_1_part')